# Expfit notes 3b: Log-transformed J & H

Instead of using constraints $c_j < 0$, we can set $c = -e^q \rightarrow q = \log(-c)$ and use

\begin{align}
E_m &= \frac{1}{n}\sum_{i=1}^n \left( y - a - \sum_{j=1}^m b_j e^{-e^{q_j} x_i} \right)^2 &&  \\
\end{align}

Shorthand:
\begin{align}
e_j = e^{-e^{q_j} x_i}
\end{align}

Derivative:
\begin{align}
\frac{\partial}{\partial q} e_j
= e_j (-x_i e^{q_j})
= -x_i e_j e^{q_j}
\end{align}

Compared to 
\begin{align}
\frac{\partial}{\partial c} e^{c x_i} = x_i e_j \quad \text{Old notation}
\end{align}
this is multiplied by $-e^q$

Chain rule

\begin{align}
\frac{\partial}{\partial x} (x_1 - f(x_2))^2 
= \frac{\partial}{\partial x_2} (f(x_2)^2 - 2 x_1 f(x_2))
= 2 (f(x_2) - x_1) \frac{\partial}{\partial x_2} f(x_2)
\end{align}

### Jacobian

Using
\begin{align}
f = a - y + \sum_{j=1}^m b_j e_j
\end{align}

\begin{align}
\frac{\partial E_m}{\partial a} &= \frac{2}{n}\sum_{i=1}^n f
\end{align}

\begin{align}
\frac{\partial E_m}{\partial b_s} &= \frac{2}{n}\sum_{i=1}^n f e_s
\end{align}

\begin{align}
\frac{\partial E_m}{\partial c_s} &= -\frac{2b_s}{n}\sum_{i=1}^n f x e_s e^{q_s}
\end{align}

### Hessian

For a
\begin{align}
\frac{\partial^2 E_2}{\partial a^2} &= 2 \\
\frac{\partial^2 E_2}{\partial b_r\,\partial a} &= \frac{2}{n}\sum^n e_r \\
\frac{\partial^2 E_2}{\partial q_r\,\partial a} &= -\frac{2b_r}{n}\sum^n x e_s e^{q_s}
\end{align}

For b
\begin{align}
\frac{\partial^2 E_2}{\partial b_r\,\partial b_s} &= \frac{2}{n}\sum_{i=1}^n e_r e_s \\
\end{align}

For b and c
\begin{align}
\frac{\partial^2 E_2}{\partial c_s\,\partial b_s} &= -\frac{2}{n}\sum_{i=1}^n (f + b_s e_s) xe_se^{q_s} \\
r \neq s, \frac{\partial^2 E_2}{\partial c_r\,\partial b_s} &= -\frac{2b_r}{n}\sum_{i=1}^n x e_r e_s e^{q_r}
\end{align}

For c
\begin{align}
r \neq s, \frac{\partial^2 E_2}{\partial c_r c_s} &= \frac{2b_rb_s}{n}\sum_{i=1}^n x^2 e_r e_s e^{q_r} e^{q_s} \\
\frac{\partial^2 E_2}{\partial c_s^2} &= \frac{2b_s}{n}\sum_{i=1}^n 
\left( (f + b_se_s) x^2 e^{2q_s} e_s - f_sxe_s e^{q_s} \right)
\end{align}

## Checking with finite differences

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def f1(x, a, b, q):
    return a + b * np.exp(-np.exp(q) * x)

a, b, q = 5, 5, 0.3
n = 100
x = np.linspace(0, 1, n)
y = f1(x, a, b, q)

a0, b0, q0 = a + 1, b - 1, q + 1
y0 = f1(x, a0, b0, q0)

In [3]:
def mse(x, y, p):
    d = len(p)
    assert (d - 1) % 2 == 0 and d > 1
    m = (d - 1) // 2
    p = np.asarray(p)    
    bs = p[1::2].reshape((m, 1))
    cs = -np.exp(p[2::2].reshape((m, 1)))
    return np.sum((p[0] - y + np.sum(bs * np.exp(cs * x), axis=0))**2) / len(x)


def mse_jac_fd(x, y, p, dp):
    e = mse(x, y, p)
    jac = np.zeros(len(p))
    p = np.array(p, dtype=float)
    for i in range(len(p)):
        q = np.copy(p)
        q[i] += dp
        jac[i] = (mse(x, y, q) - e) / dp
    return e, jac
    
    
def mse_jac_hes_fd(x, y, p, dp=1e-5):
    d = len(p)
    m = (d - 1) // 2
    mse, jac = mse_jac_fd(x, y, p, dp)
    
    hes = np.zeros((d, d))
    p = np.array(p, dtype=float)
    for i in range(len(p)):
        q = np.copy(p)
        q[i] += dp
        hes[i] = (mse_jac_fd(x, y, q, dp)[1] - jac) / dp   
    return mse, jac, hes


In [4]:
def mse_jac_hes(x, y, p):
    d = len(p)
    assert (d - 1) % 2 == 0 and d > 1
    m = (d - 1) // 2

    # Unpack
    p = np.asarray(p)
    a = p[0]
    bs = p[1::2].reshape((m, 1))        # (m, 1)
    qs = p[2::2].reshape((m, 1))        # (m, 1)
    
    # MSE
    n = len(x)
    ninv2 = 2 / n
    eqs = np.exp(qs)
    es = np.exp(-eqs * x)               # (m, n)
    bes = bs * es                       # (m, n)
    fs = a - y + np.sum(bes, axis=0)    # (n, )
    mse = np.sum(fs**2) / n

    # Jacobian
    jac = np.zeros(d)
    xes = es * x * eqs   
    jac[0] = ninv2 * np.sum(fs)
    jac[1::2] = ninv2 * np.sum(fs * es, axis=1)
    jac[2::2] = -ninv2 * np.sum(fs * xes, axis=1) * bs.T

    # Hessian
    hes = np.zeros((d, d))
    # aa, ab, ac
    hes[0, 0] = 2
    hes[0, 1::2] = hes[1::2, 0] = ninv2 * np.sum(es, axis=1)
    hes[0, 2::2] = hes[2::2, 0] = -ninv2 * np.sum(xes, axis=1) * bs.T
    for i in range(m):
        # bi^2, ci^2, and bi*ci
        hes[1 + 2 * i, 1 + 2 * i] = ninv2 * np.sum(es[i]**2)       
        hes[2 + 2 * i, 2 + 2 * i] = ninv2 * bs[i, 0] * np.sum(
            ((fs + bes[i]) * x * eqs[i] - fs) * xes[i])
        hes[1 + 2 * i, 2 + 2 * i] = hes[2 + 2 * i, 1 + 2 * i] = \
            -ninv2 * np.sum((fs + bes[i]) * xes[i])
        for j in range(i + 1, m):
            # bi*bj, ci*cj, bi*cj, bj*ci
            hes[1 + 2 * i, 1 + 2 * j] = hes[1 + 2 * j, 1 + 2 * i] = \
                ninv2 * np.sum(es[i] * es[j])
            hes[2 + 2 * i, 2 + 2 * j] = hes[2 + 2 * j, 2 + 2 * i] = \
                ninv2 * np.sum(xes[i] * xes[j]) * bs[i, 0] * bs[j, 0]
            hes[1 + 2 * i, 2 + 2 * j] = hes[2 + 2 * j, 1 + 2 * i] = \
                -ninv2 * np.sum(xes[j] * es[i]) * bs[j, 0]
            hes[2 + 2 * i, 1 + 2 * j] = hes[1 + 2 * j, 2 + 2 * i] = \
                -ninv2 * np.sum(xes[i] * es[j]) * bs[i, 0]

    return mse, jac, hes


m1, j1, h1 = mse_jac_hes_fd(x, y, (a0, b0, q0))
m2, j2, h2 = mse_jac_hes(x, y, (a0, b0, q0))
print(m1)
print(m2)
print()
print(j1)
print(j2)
print()
print(h1)
print(h2)

0.5318340993548341
0.5318340993548341

[-1.35089189 -0.35493886  1.49085924]
[-1.35090189 -0.35494026  1.49085237]

[[ 1.99999906  0.53616889 -1.90498062]
 [ 0.53616889  0.27975733 -0.16382229]
 [-1.90498062 -0.16382229  1.37402756]]
[[ 2.          0.53617099 -1.90498574]
 [ 0.53617099  0.27976106 -0.16382263]
 [-1.90498574 -0.16382263  1.374067  ]]


And now we can do a proper comparison:

In [5]:
m1, j1, h1 = mse_jac_hes(x, y, (a0, b0, q0, 3, 0.5))
m2, j2, h2 = mse_jac_hes_fd(x, y, (a0, b0, q0, 3, 0.5))
print(m1)
print(m2)
print()
print(j1)
print(j2)
print()
with np.printoptions(linewidth=120):
    print(h1)
    print(h2)

1.1252666913718103
1.1252666913718103

[ 1.59492512  0.78695572 -1.49775538  1.08083808 -1.02586013]
[ 1.59493512  0.78695712 -1.49774428  1.08084102 -1.02585303]

[[ 2.          0.53617099 -1.90498574  0.98194234 -1.77717235]
 [ 0.53617099  0.27976106 -0.91097456  0.380632   -0.33571701]
 [-1.90498574 -0.91097456  2.22062617 -0.99620258  1.72325158]
 [ 0.98194234  0.380632   -0.99620258  0.58868481 -1.10126086]
 [-1.77717235 -0.33571701  1.72325158 -1.10126086  1.42073185]]
[[ 2.00000017  0.53616889 -1.90498062  0.98194342 -1.77717396]
 [ 0.53616889  0.27975844 -0.9109713   0.38063108 -0.33572256]
 [-1.90498062 -0.9109713   2.22060148 -0.99620312  1.72324155]
 [ 0.98194342  0.38063108 -0.99620312  0.5886891  -1.10125908]
 [-1.77717396 -0.33572256  1.72324155 -1.10125908  1.42073464]]


This looks OK

In [6]:
with np.printoptions(linewidth=120):
    print((h1 - h2))

[[-1.65480742e-07  2.09799908e-06 -5.12644248e-06 -1.07945380e-06  1.61317802e-06]
 [ 2.09799908e-06  2.62530597e-06 -3.26634651e-06  9.13238412e-07  5.55455880e-06]
 [-5.12644248e-06 -3.26634651e-06  2.46869868e-05  5.36103173e-07  1.00282954e-05]
 [-1.07945380e-06  9.13238412e-07  5.36103173e-07 -4.28623501e-06 -1.77506983e-06]
 [ 1.61317802e-06  5.55455880e-06  1.00282954e-05 -1.77506983e-06 -2.79546938e-06]]


In [10]:
m1, j1, h1 = mse_jac_hes(x, y, (a0, b0, q0, 3, -7, 1.1, 2.2))
m2, j2, h2 = mse_jac_hes_fd(x, y, (a0, b0, q0, 3, -7, 1.1, 2.2))
print(m1)
print(m2)
print()
with np.printoptions(linewidth=120):
    print(j1)
    print(j2)
    print()
    print(np.abs(h1 - h2) < 5e-5)

6.180662833781528
6.180662833781528

[ 4.89883194  1.43602515 -4.41990583  4.89663664 -0.00658384  0.70968099 -0.60486045]
[ 4.89884194  1.43602655 -4.41988182  4.89664663 -0.00658387  0.70968159 -0.60485768]

[[ True  True  True  True  True  True  True]
 [ True  True  True  True  True  True  True]
 [ True  True  True  True  True  True  True]
 [ True  True  True  True  True  True  True]
 [ True  True  True  True  True  True  True]
 [ True  True  True  True  True  True  True]
 [ True  True  True  True  True  True  True]]
